# Notebook 03: Partition-Based Clustering (K-Means & Manhattan K-Medoids)
**Project**: Maternal-Child Health Profiles Across Countries Through Unsupervised Clustering

### Objectives:
1. Benchmark K-Means across $K=2\dots10$ using multiple initializations (`k-means++`, $n\_init=20$).
2. Compute WCSS/Inertia, Silhouette Scores, Davies-Bouldin Index, and Calinski-Harabasz Index.
3. Compare Euclidean K-Means with Manhattan K-Medoids (PAM) and evaluate distance metric differences.

In [ ]:
import sys
import os
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import config
from src.data_loader import get_latest_cross_sectional_dataset
from src.preprocessing import HealthDataPreprocessor
from src.clustering import run_kmeans_experiments, run_kmedoids_experiments, compare_kmeans_vs_kmedoids

### 1. K-Means Grid Search across K=2..10

In [ ]:
raw_data = get_latest_cross_sectional_dataset(min_year=2000)
preprocessor = HealthDataPreprocessor(max_missing_country_pct=35.0, imputation_strategy="median")
_, _, standardized = preprocessor.fit_transform_pipeline(raw_data, save_artifacts=False)

X = standardized[config.FEATURE_COLUMNS].values
kmeans_df, models = run_kmeans_experiments(X)
kmeans_df[["K", "WCSS", "Silhouette", "DaviesBouldin", "CalinskiHarabasz", "ClusterSizes"]]

### 2. Elbow & Silhouette Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(kmeans_df["K"], kmeans_df["WCSS"], "o-", color="#1f77b4", linewidth=2.5)
ax1.set_title("Elbow Curve (WCSS / Inertia vs K)")
ax1.set_xlabel("Number of Clusters (K)")
ax1.set_ylabel("WCSS")
ax1.grid(True)

ax2.plot(kmeans_df["K"], kmeans_df["Silhouette"], "s-", color="#2ca02c", linewidth=2.5)
ax2.set_title("Silhouette Score vs K")
ax2.set_xlabel("Number of Clusters (K)")
ax2.set_ylabel("Average Silhouette")
ax2.grid(True)
plt.tight_layout()
plt.show()

### 3. Euclidean K-Means vs Manhattan K-Medoids Comparison

In [ ]:
kmed_df, _ = run_kmedoids_experiments(X, metric="manhattan")
comp_k4 = compare_kmeans_vs_kmedoids(X, k=4)
print(f"Agreement between Euclidean K-Means and Manhattan K-Medoids (ARI): {comp_k4['ARI_Agreement']:.4f}")
print("K-Means:", comp_k4["KMeans"])
print("K-Medoids:", comp_k4["KMedoids_Manhattan"])

plt.figure(figsize=(9, 5))
plt.plot(kmeans_df["K"], kmeans_df["Silhouette"], "o-", label="Euclidean K-Means")
plt.plot(kmed_df["K"], kmed_df["Silhouette_Manhattan"], "s--", label="Manhattan K-Medoids")
plt.title("Distance Metric Comparison: Euclidean vs Manhattan Silhouette")
plt.xlabel("K")
plt.ylabel("Silhouette Score")
plt.legend()
plt.grid(True)
plt.show()